In [4]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [5]:
df = pd.read_csv("../data/processed/manali_places_enriched.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 20)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,nature,history,culture,adventure,photography,shopping,religious,family,travel_tags,interest_count
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,1,1,1,0,1,1,1,0,"culture, history, nature, photography, religio...",6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,1,1,1,0,1,0,0,1,"culture, family, history, nature, photography",5
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,0,1,0,0,1,0,0,0,"history, photography",2
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,0,0,0,0,0,0,0,0,NaN,0
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,1,1,0,0,1,0,0,1,"family, history, nature, photography",4


In [6]:
feature_columns = [
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]

feature_columns


['nature',
 'history',
 'culture',
 'adventure',
 'photography',
 'shopping',
 'religious',
 'family']

In [7]:
X = df[feature_columns].fillna(0).astype(float)

print("Feature matrix shape:", X.shape)
X.head()


Feature matrix shape: (20, 8)


,nature,history,culture,adventure,photography,shopping,religious,family
0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0
1,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [8]:
example_place = df.iloc[0]

print("Place:", example_place["name"])
print("Vector:", example_place[feature_columns].tolist())


Place: Hadimba Devi Temple
Vector: [np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0)]


In [9]:
user_preferences = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 0.0,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.0
}

user_vector = np.array([
    user_preferences[feature]
    for feature in feature_columns
]).reshape(1, -1)

print("User vector:")
print(user_vector)


User vector:
[[1. 0. 0. 0. 1. 0. 0. 0.]]


In [10]:
similarity_scores = cosine_similarity(
    user_vector,
    X
).flatten()

df["interest_similarity"] = similarity_scores

df[[
    "name",
    "interest_similarity"
]].sort_values(
    "interest_similarity",
    ascending=False
).head(10)


,name,interest_similarity
8,Lama Dugh Trek Start Point,0.707107
6,Manali View Point,0.707107
5,Van Vihar National Park,0.707107
4,Jogini Falls,0.707107
1,Old Manali snow point,0.632456
0,Hadimba Devi Temple,0.577350
10,Kharma valley,0.500000
2,Nehru Kund,0.500000
14,Baror Parsha Waterfall,0.500000
18,Gulaba Viewpoint,0.408248


In [11]:
rating_scaler = MinMaxScaler()

df["rating_score"] = rating_scaler.fit_transform(
    df[["rating"]]
).flatten()

df[["name", "rating", "rating_score"]].head()


,name,rating,rating_score
0,Hadimba Devi Temple,4.6,0.777778
1,Old Manali snow point,4.6,0.777778
2,Nehru Kund,4.4,0.555556
3,Kullu Manali River rafting,4.5,0.666667
4,Jogini Falls,4.6,0.777778


In [12]:
df["log_reviews"] = np.log1p(
    df["reviews"].clip(lower=0)
)

popularity_scaler = MinMaxScaler()

df["popularity_score"] = popularity_scaler.fit_transform(
    df[["log_reviews"]]
).flatten()

df[[
    "name",
    "reviews",
    "log_reviews",
    "popularity_score"
]].sort_values(
    "reviews",
    ascending=False
).head(10)


,name,reviews,log_reviews,popularity_score
0,Hadimba Devi Temple,49688,10.813539,1.000000
4,Jogini Falls,10842,9.291275,0.817225
5,Van Vihar National Park,9050,9.110631,0.795536
2,Nehru Kund,7767,8.957768,0.777182
19,Manali Bazaar,3991,8.292048,0.697250
18,Gulaba Viewpoint,3576,8.182280,0.684071
7,Rahala Waterfalls,797,6.682109,0.503949
14,Baror Parsha Waterfall,489,6.194405,0.445391
1,Old Manali snow point,428,6.061457,0.429428
8,Lama Dugh Trek Start Point,297,5.697093,0.385680


In [13]:
df["final_score"] = (
    0.70 * df["interest_similarity"]
    + 0.20 * df["rating_score"]
    + 0.10 * df["popularity_score"]
)

recommendation_columns = [
    "name",
    "category",
    "rating",
    "reviews",
    "interest_similarity",
    "rating_score",
    "popularity_score",
    "final_score"
]

df[recommendation_columns].sort_values(
    "final_score",
    ascending=False
).head(10)


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
4,Jogini Falls,Tourist attraction,4.6,10842,0.707107,0.777778,0.817225,0.732253
8,Lama Dugh Trek Start Point,Tourist attraction,4.6,297,0.707107,0.777778,0.385680,0.689098
6,Manali View Point,Tourist attraction,4.6,87,0.707107,0.777778,0.239227,0.674453
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,0.577350,0.777778,1.000000,0.659701
1,Old Manali snow point,Tourist attraction,4.6,428,0.632456,0.777778,0.429428,0.641217
5,Van Vihar National Park,Tourist attraction,4.2,9050,0.707107,0.333333,0.795536,0.641195
10,Kharma valley,Tourist attraction,4.8,143,0.500000,1.000000,0.298357,0.579836
14,Baror Parsha Waterfall,Tourist attraction,4.7,489,0.500000,0.888889,0.445391,0.572317
2,Nehru Kund,Tourist attraction,4.4,7767,0.500000,0.555556,0.777182,0.538829
18,Gulaba Viewpoint,Tourist attraction,4.5,3576,0.408248,0.666667,0.684071,0.487514


In [14]:
def recommend_places(user_preferences, top_n=5):
    """
    Recommend places based on user travel preferences.

    Parameters
    ----------
    user_preferences : dict
        Preference strength for each recommendation feature.
    top_n : int
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Ranked recommendations.
    """

    missing_features = [
        feature
        for feature in feature_columns
        if feature not in user_preferences
    ]

    if missing_features:
        raise ValueError(
            f"Missing preference features: {missing_features}"
        )

    user_vector = np.array([
        float(user_preferences[feature])
        for feature in feature_columns
    ]).reshape(1, -1)

    similarity = cosine_similarity(
        user_vector,
        X
    ).flatten()

    result = df.copy()
    result["interest_similarity"] = similarity

    result["final_score"] = (
        0.70 * result["interest_similarity"]
        + 0.20 * result["rating_score"]
        + 0.10 * result["popularity_score"]
    )

    return result.sort_values(
        "final_score",
        ascending=False
    )[recommendation_columns].head(top_n).reset_index(drop=True)


In [15]:
user_1 = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 0.0,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.0
}

recommend_places(user_1, top_n=5)


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
0,Jogini Falls,Tourist attraction,4.6,10842,0.707107,0.777778,0.817225,0.732253
1,Lama Dugh Trek Start Point,Tourist attraction,4.6,297,0.707107,0.777778,0.385680,0.689098
2,Manali View Point,Tourist attraction,4.6,87,0.707107,0.777778,0.239227,0.674453
3,Hadimba Devi Temple,Tourist attraction,4.6,49688,0.577350,0.777778,1.000000,0.659701
4,Old Manali snow point,Tourist attraction,4.6,428,0.632456,0.777778,0.429428,0.641217


In [16]:
user_2 = {
    "nature": 0.0,
    "history": 1.0,
    "culture": 0.8,
    "adventure": 0.0,
    "photography": 0.3,
    "shopping": 0.0,
    "religious": 1.0,
    "family": 0.2
}

recommend_places(user_2, top_n=5)


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,0.760407,0.777778,1.000000,0.787841
1,Shiv Mahadev Temple,Tourist attraction,4.6,270,0.849719,0.777778,0.374277,0.787786
2,Old Manali snow point,Tourist attraction,4.6,428,0.618021,0.777778,0.429428,0.631113
3,Atal Bihari statue,Tourist attraction,4.5,74,0.600842,0.666667,0.220034,0.575926
4,Nehru Kund,Tourist attraction,4.4,7767,0.552317,0.555556,0.777182,0.575451


In [17]:
user_3 = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 1.0,
    "photography": 0.6,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.0
}

recommend_places(user_3, top_n=5)


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
0,Lama Dugh Trek Start Point,Tourist attraction,4.6,297,0.650945,0.777778,0.385680,0.649785
1,Jogini Falls,Tourist attraction,4.6,10842,0.520756,0.777778,0.817225,0.601807
2,Hadimba Devi Temple,Tourist attraction,4.6,49688,0.425195,0.777778,1.000000,0.553192
3,Kharma valley,Tourist attraction,4.8,143,0.460287,1.000000,0.298357,0.552037
4,Baror Parsha Waterfall,Tourist attraction,4.7,489,0.460287,0.888889,0.445391,0.544518


In [18]:
user_4 = {
    "nature": 0.7,
    "history": 0.2,
    "culture": 0.4,
    "adventure": 0.2,
    "photography": 0.4,
    "shopping": 0.2,
    "religious": 0.1,
    "family": 1.0
}

recommend_places(user_4, top_n=5)


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
0,Jogini Falls,Tourist attraction,4.6,10842,0.825652,0.777778,0.817225,0.815234
1,Old Manali snow point,Tourist attraction,4.6,428,0.866918,0.777778,0.429428,0.805341
2,Gulaba Viewpoint,Tourist attraction,4.5,3576,0.787575,0.666667,0.684071,0.753043
3,Rahala Waterfalls,Tourist attraction,4.5,797,0.787575,0.666667,0.503949,0.735031
4,Van Vihar National Park,Tourist attraction,4.2,9050,0.825652,0.333333,0.795536,0.724177


In [19]:
recommendations = recommend_places(user_1, top_n=10)

recommendations


,name,category,rating,reviews,interest_similarity,rating_score,popularity_score,final_score
0,Jogini Falls,Tourist attraction,4.6,10842,0.707107,0.777778,0.817225,0.732253
1,Lama Dugh Trek Start Point,Tourist attraction,4.6,297,0.707107,0.777778,0.385680,0.689098
2,Manali View Point,Tourist attraction,4.6,87,0.707107,0.777778,0.239227,0.674453
3,Hadimba Devi Temple,Tourist attraction,4.6,49688,0.577350,0.777778,1.000000,0.659701
4,Old Manali snow point,Tourist attraction,4.6,428,0.632456,0.777778,0.429428,0.641217
5,Van Vihar National Park,Tourist attraction,4.2,9050,0.707107,0.333333,0.795536,0.641195
6,Kharma valley,Tourist attraction,4.8,143,0.500000,1.000000,0.298357,0.579836
7,Baror Parsha Waterfall,Tourist attraction,4.7,489,0.500000,0.888889,0.445391,0.572317
8,Nehru Kund,Tourist attraction,4.4,7767,0.500000,0.555556,0.777182,0.538829
9,Gulaba Viewpoint,Tourist attraction,4.5,3576,0.408248,0.666667,0.684071,0.487514


In [20]:
output_path = "../data/processed/manali_recommendation_features.csv"

df.to_csv(output_path, index=False)

print(f"✅ Recommendation feature dataset saved to: {output_path}")


✅ Recommendation feature dataset saved to: ../data/processed/manali_recommendation_features.csv
